# Data Prep -- Fine-tuning Pair Extraction

Builds `df_finetune_pairs` from `tdr-experiments/data/TDR_train/`: one
(undiacritized `Train`, original `Target`) sentence pair per sentence that
contains at least one target-homograph match -- diacritized variant or the
bare/normalized form, either counts.

`homographs`, `normalize_word`, `HOMOGRAPH_PATTERN`, `strip_invisible`,
`read_text_clean` are copied verbatim from `analysis/05_iaa_and_tdr_stats.ipynb`
(not imported across notebooks).

This pass is extraction + a sanity-check report only -- no train/val split
or JSONL export yet.

## Config

`TRAIN_DIR` resolves to `tdr-experiments/data/TDR_train/` regardless of
whether this notebook's kernel working directory is the project root or
`tdr-experiments/` itself.

In [ ]:
from pathlib import Path

_candidates = [
    Path("tdr-experiments") / "data" / "TDR_train",
    Path("data") / "TDR_train",
]
TRAIN_DIR = next((p for p in _candidates if p.is_dir()), None)
assert TRAIN_DIR is not None, f"TDR_train not found in any of: {_candidates}"

print(f"TRAIN_DIR: {TRAIN_DIR.resolve()}")

## Shared setup: homographs, normalization, alternation regex

Copied verbatim from `analysis/05_iaa_and_tdr_stats.ipynb`.

In [ ]:
import re

import pandas as pd

homographs = {
    "kaya": ["kayâ", "káya"],
    "bilang": ["biláng", "bílang"],
    "nasa": ["nasà", "nása"],
    "lamang": ["lamáng", "lámang"],
    "kasama": ["kasamá", "kasáma", "kasamâ"],
    "bagong": ["bagóng", "bágong"],
    "buhay": ["buháy", "búhay"],
    "dating": ["datíng", "dáting"],
    "silang": ["siláng", "sílang"],
    "pangulong": ["pangulóng", "pangúlong"],
    "tayo": ["tayô", "táyo"],
    "gabi": ["gabí", "gábi"],
    "kabilang": ["kabílang", "kabilâng"],
    "taong": ["taóng", "táong"],
    "patuloy": ["patulóy", "patúloy"],
    "pamahalaan": ["pámahalaán", "pámahaláan"],
    "lalaki": ["laláki", "lalakí"],
    "tapos": ["tapós", "tápos"],
    "pala": ["pála", "palá"],
    "baka": ["bakâ", "báka"],
    "paano": ["paanó", "paáno"],
    "huling": ["húling", "hulíng"],
    "sakit": ["sakít", "sákit"],
    "panalo": ["panaló", "panálo"],
    "kaibigan": ["kaibigán", "kaibígan"],
    "sasakyan": ["sasakyán", "sasákyan"],
    "ika": ["ikà", "ikâ", "iká"],
    "madaling": ["madalîng", "madáling"],
    "galing": ["galíng", "gáling"],
    "kita": ["kitá", "kíta"],
    "gamit": ["gamít", "gámit"],
    "huli": ["húli", "hulí"],
    "bukas": ["bukás", "búkas"],
    "kilala": ["kilalá", "kilála"],
    "galit": ["galít", "gálit"],
    "takot": ["takót", "tákot"],
    "puno": ["punô", "punò"],
    "pagod": ["pagód", "págod"],
    "abala": ["abalá", "abála"],
    "butas": ["butás", "bútas"],
}

assert len(homographs) == 40
print(f"Loaded {len(homographs)} target homographs.")

_DIACRITIC_MAP = {}
for _base, _marked in [("a", "áàâ"), ("e", "éèê"), ("i", "íìî"), ("o", "óòô"), ("u", "úùû")]:
    for _ch in _marked:
        _DIACRITIC_MAP[_ch] = _base
        _DIACRITIC_MAP[_ch.upper()] = _base.upper()
_DIACRITIC_TABLE = str.maketrans(_DIACRITIC_MAP)


def normalize_word(word: str) -> str:
    """Strips diacritics and lowercases -- used ONLY for matching a token
    against the `homographs` dict keys. Original casing/diacritics are
    preserved separately wherever a token's actual text is recorded."""
    return word.translate(_DIACRITIC_TABLE).lower()


# All known surface forms per homograph -- its bare (undiacritized) form
# plus every diacritized variant -- combined into a single alternation
# regex, so each file only needs one finditer pass. Longest forms first so
# a shorter form (e.g. "ika") can't win the alternation over a longer one
# that contains it.
_ALL_FORMS = set()
for _h, _variants in homographs.items():
    _ALL_FORMS.add(_h)
    _ALL_FORMS.update(_variants)

HOMOGRAPH_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(f) for f in sorted(_ALL_FORMS, key=len, reverse=True)) + r")\b",
    re.IGNORECASE,
)

INVISIBLE_CHARS = "\xad\ufeff\u200b\u200c\u200d\u2060"
INVISIBLE_TABLE = str.maketrans("", "", INVISIBLE_CHARS)


def strip_invisible(text: str) -> str:
    return text.translate(INVISIBLE_TABLE)


def read_text_clean(path: Path) -> str:
    return strip_invisible(path.read_text(encoding="utf-8"))

## Step 1: Sentence-splitting helper

In [ ]:
SENTENCE_SPLIT_PATTERN = re.compile(r'[.!?]+|\n')

def split_sentences(text: str) -> list[str]:
    """Splits on sentence-ending punctuation OR newlines; returns
    non-empty, whitespace-stripped segments in order."""
    return [seg.strip() for seg in SENTENCE_SPLIT_PATTERN.split(text) if seg.strip()]

## Step 2: Extract fine-tuning pairs from TDR_train

Walks every file, splits into sentences, and keeps a sentence if it
contains at least one target-homograph match -- diacritized variant or
the bare/normalized form, either qualifies.

`Train` strips diacritics from the WHOLE sentence via the
`_DIACRITIC_TABLE` character-translation table only (no `.lower()`), so
casing (sentence-initial capitals, proper nouns) is preserved -- only the
diacritic marks themselves are removed. `Target` is the sentence
unchanged.

`Sentence_Index` is this sentence's 0-based position among ALL sentences
`split_sentences` found in its file (not just the qualifying ones), so it
can be used to relocate the sentence in its source file later.

In [ ]:
from collections import Counter

train_files = sorted(TRAIN_DIR.glob("*.txt"))

rows = []
homograph_pair_counts = Counter()
total_sentences = 0

for path in train_files:
    text = read_text_clean(path)
    sentences = split_sentences(text)
    total_sentences += len(sentences)

    for idx, sentence in enumerate(sentences):
        homographs_in_sentence = {
            normalize_word(match.group(0))
            for match in HOMOGRAPH_PATTERN.finditer(sentence)
        }
        if not homographs_in_sentence:
            continue

        rows.append({
            "Filename": path.name,
            "Sentence_Index": idx,
            "Train": sentence.translate(_DIACRITIC_TABLE),
            "Target": sentence,
        })
        for h in homographs_in_sentence:
            homograph_pair_counts[h] += 1

df_finetune_pairs = pd.DataFrame(rows, columns=["Filename", "Sentence_Index", "Train", "Target"])

## Step 3: Report (extraction sanity check -- no export yet)

In [ ]:
print(f"Total files processed:       {len(train_files)}")
print(f"Total sentences scanned:     {total_sentences}")
print(f"Total qualifying pairs:      {len(df_finetune_pairs)}")

print("\n5 example pairs:")
for _, row in df_finetune_pairs.head(5).iterrows():
    print(f"\n[{row['Filename']} #{row['Sentence_Index']}]")
    print(f"  Train:  {row['Train']}")
    print(f"  Target: {row['Target']}")

print("\nQualifying pairs per homograph (sentence-level presence, most to fewest):")
for homograph, count in homograph_pair_counts.most_common():
    print(f"  {homograph:15s} {count}")

if homograph_pair_counts:
    most = homograph_pair_counts.most_common(1)[0]
    fewest = homograph_pair_counts.most_common()[-1]
    print(f"\nMost common:  {most[0]} ({most[1]} sentence(s))")
    print(f"Fewest:       {fewest[0]} ({fewest[1]} sentence(s))")

missing_homographs = sorted(set(homographs) - set(homograph_pair_counts))
if missing_homographs:
    print(f"\nHomograph(s) with ZERO qualifying training sentences: {missing_homographs}")

## Step 4 (revised): Per-(homograph, variant) 90/10 split, smallest group first

Per-homograph splitting (previous version of this cell) still lumps every
variant of a homograph together -- a rare variant like `ikâ` could still
end up with zero val examples if it's a small slice of a
larger-total homograph. This version splits at the (homograph, variant)
level instead: `Primary_Variant` is the ACTUAL matched text (lowercased)
of the `Primary_Homograph` occurrence in `Target` -- one of its
diacritized variants, or the bare/normalized form itself if that instance
was left undiacritized (e.g. `("dating", "dáting")`, `("dating",
"datíng")`, or `("dating", "dating")` for the bare case).

Every (homograph, variant) group across all 40 homographs is pooled
together and sorted ascending by count -- smallest group first overall,
regardless of which homograph it belongs to, so a 2-sentence variant gets
processed before a several-thousand-sentence one even if they belong to
different homographs. Each group is shuffled (fixed seed 42) and split
~90/10 independently, `n_val = max(1, round(0.1 * len(group)))` whenever
it has 2+ sentences; a group with exactly 1 sentence is a "singleton" --
forced into train only, recorded rather than silently dropped.

This val split is ONLY for monitoring training loss / early stopping
during fine-tuning -- it is a held-out slice of `df_finetune_pairs`
(`TDR_train`), NOT the same as `TDR_test`, and must not be confused with
or used as the evaluation set.

In [ ]:
RANDOM_SEED = 42


def primary_match_info(target: str):
    match = HOMOGRAPH_PATTERN.search(target)
    if match is None:
        return None, None
    token = match.group(0)
    return normalize_word(token), token.lower()


_primary_info = df_finetune_pairs["Target"].apply(primary_match_info)
df_finetune_pairs["Primary_Homograph"] = _primary_info.apply(lambda t: t[0])
df_finetune_pairs["Primary_Variant"] = _primary_info.apply(lambda t: t[1])
# Every row already matched HOMOGRAPH_PATTERN at least once in Step 2's
# qualification check, so neither of these should actually be null.
assert df_finetune_pairs["Primary_Homograph"].notna().all(), "row(s) with no Primary_Homograph found"
assert df_finetune_pairs["Primary_Variant"].notna().all(), "row(s) with no Primary_Variant found"

group_counts = (
    df_finetune_pairs.groupby(["Primary_Homograph", "Primary_Variant"])
    .size()
    .reset_index(name="Count")
    .sort_values("Count", ascending=True)
)
group_order = list(group_counts[["Primary_Homograph", "Primary_Variant"]].itertuples(index=False, name=None))

print(f"Total distinct (homograph, variant) groups: {len(group_order)}")

train_rows = []
val_rows = []
singleton_groups = []

for h, v in group_order:
    subset = df_finetune_pairs.loc[
        (df_finetune_pairs["Primary_Homograph"] == h) & (df_finetune_pairs["Primary_Variant"] == v)
    ]
    subset = subset.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n = len(subset)

    if n >= 2:
        n_val = max(1, round(0.1 * n))
        val_rows.append(subset.iloc[:n_val])
        train_rows.append(subset.iloc[n_val:])
    else:
        train_rows.append(subset)
        singleton_groups.append((h, v))

df_train = pd.concat(train_rows, ignore_index=True) if train_rows else df_finetune_pairs.iloc[0:0].copy()
df_val = pd.concat(val_rows, ignore_index=True) if val_rows else df_finetune_pairs.iloc[0:0].copy()

print(f"\nTotal pairs:      {len(df_finetune_pairs)}")
print(f"Train split:      {len(df_train)} ({len(df_train) / len(df_finetune_pairs):.1%})")
print(f"Val split:        {len(df_val)} ({len(df_val) / len(df_finetune_pairs):.1%})")
print(f"train + val == total: {len(df_train) + len(df_val) == len(df_finetune_pairs)}")
print(f"Singleton (1-sentence) groups, train-only: {len(singleton_groups)}")
if singleton_groups:
    for h, v in singleton_groups:
        print(f"  ({h}, {v})")

## Step 5: Format as instruction records

One fixed instruction string, `input` = the undiacritized `Train`
sentence, `output` = the diacritized `Target` sentence.

In [ ]:
INSTRUCTION = "Modify the homographs in this Tagalog sentence by adding the correct diacritic marks (tuldik)."


def to_records(df):
    return [
        {"instruction": INSTRUCTION, "input": row["Train"], "output": row["Target"]}
        for _, row in df.iterrows()
    ]


train_records = to_records(df_train)
val_records = to_records(df_val)

print(f"train_records: {len(train_records)}")
print(f"val_records:   {len(val_records)}")
print("\nSample record:")
print(train_records[0])

## Step 6: Export

`ensure_ascii=False` so diacritics are written as literal UTF-8 characters
rather than `\uXXXX` escapes.

In [ ]:
import json

_candidates_data_dir = [
    Path("tdr-experiments") / "data",
    Path("data"),
]
DATA_DIR = next((p for p in _candidates_data_dir if p.is_dir()), None)
assert DATA_DIR is not None, f"tdr-experiments/data not found in any of: {_candidates_data_dir}"

TRAIN_JSONL = DATA_DIR / "finetune_train.jsonl"
VAL_JSONL = DATA_DIR / "finetune_val.jsonl"


def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


write_jsonl(train_records, TRAIN_JSONL)
write_jsonl(val_records, VAL_JSONL)

print(f"Wrote {len(train_records)} record(s) to {TRAIN_JSONL.resolve()}")
print(f"Wrote {len(val_records)} record(s) to {VAL_JSONL.resolve()}")

## Step 7: Report

Re-reads both JSONL files back from disk (rather than printing the
in-memory records) to confirm the write/read round-trip is clean and
diacritics survive intact.

In [ ]:
train_count = sum(1 for _ in open(TRAIN_JSONL, encoding="utf-8"))
val_count = sum(1 for _ in open(VAL_JSONL, encoding="utf-8"))

print(f"Final train count: {train_count}")
print(f"Final val count:   {val_count}")
print(f"train + val:       {train_count + val_count} (should be {len(df_finetune_pairs)})")
print(f"Matches df_finetune_pairs total: {train_count + val_count == len(df_finetune_pairs)}")

print(f"\nTotal distinct (homograph, variant) groups: {len(group_order)}")

train_counts_by_group = df_train.groupby(["Primary_Homograph", "Primary_Variant"]).size()
val_counts_by_group = df_val.groupby(["Primary_Homograph", "Primary_Variant"]).size()

print("\nPer-group breakdown (ascending total-count order, smallest group first):")
print(f"{'Homograph':<14}{'Variant':<14}{'Is_Bare':>8}{'Total':>8}{'Train':>8}{'Val':>8}{'Val_Pct':>9}")
bare_group_rows = []
for h, v in group_order:
    t = int(train_counts_by_group.get((h, v), 0))
    val_n = int(val_counts_by_group.get((h, v), 0))
    total = t + val_n
    is_bare = (v == h)
    pct = f"{val_n / total:.1%}" if total else "n/a"
    print(f"{h:<14}{v:<14}{str(is_bare):>8}{total:>8}{t:>8}{val_n:>8}{pct:>9}")
    if is_bare:
        bare_group_rows.append((h, v, total, t, val_n))

print(f"\nSingleton groups (exactly 1 sentence, forced into train, 0 val): {len(singleton_groups)}")
if singleton_groups:
    for h, v in singleton_groups:
        print(f"  ({h}, {v})")

print(f"\nBARE/undiacritized groups: {len(bare_group_rows)}")
print(f"{'Homograph':<14}{'Total':>8}{'Train':>8}{'Val':>8}{'Val_Pct':>9}")
for h, v, total, t, val_n in bare_group_rows:
    pct = f"{val_n / total:.1%}" if total else "n/a"
    print(f"{h:<14}{total:>8}{t:>8}{val_n:>8}{pct:>9}")
under_represented_bare = [(h, v) for h, v, total, t, val_n in bare_group_rows if total >= 2 and val_n == 0]
if under_represented_bare:
    print(f"\nWARNING: bare group(s) with 2+ instances but ZERO val representation: {under_represented_bare}")
else:
    print("\nEvery bare group with 2+ instances has at least 1 in val -- confirmed.")


def human_size(num_bytes):
    if num_bytes >= 1024 * 1024:
        return f"{num_bytes / (1024 * 1024):.2f} MB"
    return f"{num_bytes / 1024:.2f} KB"


print(f"\n{TRAIN_JSONL.name}: {human_size(TRAIN_JSONL.stat().st_size)}")
print(f"{VAL_JSONL.name}:   {human_size(VAL_JSONL.stat().st_size)}")

print("\n3 full example records re-read from finetune_train.jsonl:")
with open(TRAIN_JSONL, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        record = json.loads(line)
        print(f"\nRecord {i}:")
        print(record)

## Step 8: Extract test instances from TDR_test (diacritized only, fully normalized context)

`TEST_DIR` resolves the same way `TRAIN_DIR` does in Config, above.

Walks every file, splits into sentences, and for each sentence walks
EVERY `HOMOGRAPH_PATTERN` match in order (diacritized and bare alike, so
`Occurrence_Index` stays a true 1-based position for that homograph within
the sentence, same "count every occurrence, not just the qualifying ones"
convention used everywhere else in this project) -- but only emits a test
instance for matches that ARE diacritized (`token.lower() !=
normalize_word(token)`). Intentionally-undiacritized instances are
excluded entirely from this eval set.

Each qualifying occurrence gets its OWN separate row -- a sentence with 2
different homograph occurrences produces 2 instances, and a sentence with
2 occurrences of the SAME homograph also produces 2 instances, one per
position, each asking about that specific occurrence.

`Context_Sentence` strips diacritics from the WHOLE sentence (every word,
not just the occurrence in question) via the `_DIACRITIC_TABLE`
character-translation table -- the same full-sentence stripping Step 2
uses for `Train`, applied here to `TDR_test`. Because it's whole-sentence
normalization rather than single-occurrence masking, every row that comes
from the same (Filename, Sentence_Index) shares an IDENTICAL
`Context_Sentence` -- only `Homograph`/`Gold_Variant`/`Id` differ per
occurrence (verified explicitly in Step 9 below). `Id` is a convenience
string built from the other columns
(`Filename_SentenceIndex_Homograph_OccurrenceIndex`);
`df_test_instances` also keeps the components as separate columns.

In [ ]:
_test_candidates = [
    Path("tdr-experiments") / "data" / "TDR_test",
    Path("data") / "TDR_test",
]
TEST_DIR = next((p for p in _test_candidates if p.is_dir()), None)
assert TEST_DIR is not None, f"TDR_test not found in any of: {_test_candidates}"
print(f"TEST_DIR: {TEST_DIR.resolve()}")

test_files = sorted(TEST_DIR.glob("*.txt"))

test_rows = []
test_total_sentences = 0

for path in test_files:
    text = read_text_clean(path)
    sentences = split_sentences(text)
    test_total_sentences += len(sentences)

    for sent_idx, sentence in enumerate(sentences):
        context_sentence = sentence.translate(_DIACRITIC_TABLE)

        occurrence_counters = {}
        for match in HOMOGRAPH_PATTERN.finditer(sentence):
            token = match.group(0)
            homograph = normalize_word(token)
            occurrence_counters[homograph] = occurrence_counters.get(homograph, 0) + 1
            occurrence_idx = occurrence_counters[homograph]

            if token.lower() == homograph:
                continue  # bare -- excluded from this diacritized-only eval set

            test_rows.append({
                "Id": f"{path.name}_{sent_idx}_{homograph}_{occurrence_idx}",
                "Filename": path.name,
                "Sentence_Index": sent_idx,
                "Homograph": homograph,
                "Occurrence_Index": occurrence_idx,
                "Gold_Variant": token.lower(),
                "Context_Sentence": context_sentence,
            })

df_test_instances = pd.DataFrame(test_rows, columns=[
    "Id", "Filename", "Sentence_Index", "Homograph", "Occurrence_Index", "Gold_Variant", "Context_Sentence",
])

## Step 9: Verify multi-occurrence sentences produce multiple instances

Groups `df_test_instances` by (Filename, Sentence_Index) and confirms a
sentence with 2+ qualifying occurrences really does produce 2+ separate
rows, each with a different `Homograph`/`Gold_Variant`/`Id` but an
IDENTICAL `Context_Sentence` (since context is now whole-sentence
normalization, not per-occurrence masking).

In [ ]:
group_sizes = df_test_instances.groupby(["Filename", "Sentence_Index"]).size()

print("Distribution of instances-per-sentence:")
print(f"  Sentences with exactly 1 instance: {(group_sizes == 1).sum()}")
print(f"  Sentences with exactly 2 instances: {(group_sizes == 2).sum()}")
print(f"  Sentences with 3+ instances:        {(group_sizes >= 3).sum()}")
print(f"  Total sentences represented:        {len(group_sizes)}")
print(f"  Total instances (sanity check):     {group_sizes.sum()} (should be {len(df_test_instances)})")

multi_groups = group_sizes[group_sizes >= 2].index.tolist()
print(f"\n2 example (Filename, Sentence_Index) groups with 2+ instances:")
for filename, sent_idx in multi_groups[:2]:
    sub = df_test_instances.loc[
        (df_test_instances["Filename"] == filename) & (df_test_instances["Sentence_Index"] == sent_idx)
    ]
    print(f"\n=== {filename} #{sent_idx} ({len(sub)} instance(s)) ===")
    context_values = sub["Context_Sentence"].unique()
    print(f"Context_Sentence identical across all rows in this group: {len(context_values) == 1}")
    print(f"  Context_Sentence: {context_values[0]}")
    for _, row in sub.iterrows():
        print(f"  -> Id={row['Id']}  Homograph={row['Homograph']}  Gold_Variant={row['Gold_Variant']}")

## Step 10: Report only -- no choices/prompts yet

Extraction sanity check. 4-choice distractor sets and final prompt
assembly are the next step, once this extraction (and the
multi-occurrence verification in Step 9) is confirmed correct.

In [ ]:
print(f"Total files processed:   {len(test_files)}")
print(f"Total sentences scanned: {test_total_sentences}")
print(f"Total test instances (diacritized-only): {len(df_test_instances)}")

test_per_homograph = df_test_instances["Homograph"].value_counts()

print("\nPer-homograph count (most to fewest):")
for homograph, count in test_per_homograph.items():
    print(f"  {homograph:15s} {count}")

sparse_homographs = test_per_homograph[test_per_homograph < 5]
missing_homographs = sorted(set(homographs) - set(test_per_homograph.index))
print(f"\nHomograph(s) with FEWER THAN 5 test instances (flagged, not excluded):")
if len(sparse_homographs) or missing_homographs:
    for homograph, count in sparse_homographs.items():
        print(f"  {homograph:15s} {count}")
    for homograph in missing_homographs:
        print(f"  {homograph:15s} 0 (no diacritized instances found at all)")
else:
    print("  None -- every homograph has 5+ instances.")

print("\n5 example instances:")
for _, row in df_test_instances.head(5).iterrows():
    print(f"\n[{row['Id']}]")
    print(f"  Homograph:        {row['Homograph']}")
    print(f"  Gold_Variant:     {row['Gold_Variant']}")
    print(f"  Context_Sentence: {row['Context_Sentence']}")

## Step 11: Hardcoded, manually-verified 4-choice sets

`CHOICE_MAP` is pasted verbatim (already reviewed and approved) -- not
regenerated. For the 38 homographs with 2 known diacritized variants
(from `homographs`), the 4 choices are `[variant_1, variant_2,
bare_form, fake_generated]`. For `ika`/`kasama` (the 2 homographs with 3
known variants), the 4 choices are `[variant_1, variant_2, variant_3,
bare_form]` -- no fake slot needed since 3 real variants + bare already
fills 4.

`CHOICE_SOURCE_MAP` labels each of the 4 positions per homograph
("real_variant" / "bare_form" / "fake_generated") for traceability in
later error-breakdown analysis: the first `len(homographs[h])` positions
are `"real_variant"`, then whichever remaining position equals `h` itself
is `"bare_form"`, and any position left over after that is
`"fake_generated"`.

In [ ]:
CHOICE_MAP = {
    "kaya": ["kayâ", "káya", "kaya", "kayà"],
    "bilang": ["biláng", "bílang", "bilang", "bilàng"],
    "nasa": ["nasà", "nása", "nasa", "nàsa"],
    "lamang": ["lamáng", "lámang", "lamang", "lamâng"],
    "kasama": ["kasamá", "kasáma", "kasamâ", "kasama"],
    "bagong": ["bagóng", "bágong", "bagong", "bagông"],
    "buhay": ["buháy", "búhay", "buhay", "buhây"],
    "dating": ["datíng", "dáting", "dating", "datìng"],
    "silang": ["siláng", "sílang", "silang", "silàng"],
    "pangulong": ["pangulóng", "pangúlong", "pangulong", "pàngulong"],
    "tayo": ["tayô", "táyo", "tayo", "tâyo"],
    "gabi": ["gabí", "gábi", "gabi", "gabì"],
    "kabilang": ["kabílang", "kabilâng", "kabilang", "kabilàng"],
    "taong": ["taóng", "táong", "taong", "taòng"],
    "patuloy": ["patulóy", "patúloy", "patuloy", "patulôy"],
    "pamahalaan": ["pámahalaán", "pámahaláan", "pamahalaan", "pamahalàan"],
    "lalaki": ["laláki", "lalakí", "lalaki", "làlaki"],
    "tapos": ["tapós", "tápos", "tapos", "tapòs"],
    "pala": ["pála", "palá", "pala", "palâ"],
    "baka": ["bakâ", "báka", "baka", "bâka"],
    "paano": ["paanó", "paáno", "paano", "paanô"],
    "huling": ["húling", "hulíng", "huling", "hûling"],
    "sakit": ["sakít", "sákit", "sakit", "sàkit"],
    "panalo": ["panaló", "panálo", "panalo", "pânalo"],
    "kaibigan": ["kaibigán", "kaibígan", "kaibigan", "kaibigàn"],
    "sasakyan": ["sasakyán", "sasákyan", "sasakyan", "sâsakyan"],
    "ika": ["ikà", "ikâ", "iká", "ika"],
    "madaling": ["madalîng", "madáling", "madaling", "mâdaling"],
    "galing": ["galíng", "gáling", "galing", "galîng"],
    "kita": ["kitá", "kíta", "kita", "kîta"],
    "gamit": ["gamít", "gámit", "gamit", "gàmit"],
    "huli": ["húli", "hulí", "huli", "hùli"],
    "bukas": ["bukás", "búkas", "bukas", "bukâs"],
    "kilala": ["kilalá", "kilála", "kilala", "kilalâ"],
    "galit": ["galít", "gálit", "galit", "gàlit"],
    "takot": ["takót", "tákot", "takot", "takòt"],
    "puno": ["punô", "punò", "puno", "punó"],
    "pagod": ["pagód", "págod", "pagod", "pâgod"],
    "abala": ["abalá", "abála", "abala", "abalà"],
    "butas": ["butás", "bútas", "butas", "bùtas"],
}

assert len(CHOICE_MAP) == 40
assert all(len(v) == 4 for v in CHOICE_MAP.values())
print(f"CHOICE_MAP: {len(CHOICE_MAP)} homographs, 4 choices each -- verified.")

CHOICE_SOURCE_MAP = {}
for h, choices in CHOICE_MAP.items():
    n_real = len(homographs[h])
    labels = []
    for i, choice in enumerate(choices):
        if i < n_real:
            labels.append("real_variant")
        elif choice == h:
            labels.append("bare_form")
        else:
            labels.append("fake_generated")
    CHOICE_SOURCE_MAP[h] = labels

# Sanity check: exactly one "bare_form" per homograph, and the 2 three-
# variant homographs (ika, kasama) have zero "fake_generated" slots.
assert all(labels.count("bare_form") == 1 for labels in CHOICE_SOURCE_MAP.values())
assert CHOICE_SOURCE_MAP["ika"].count("fake_generated") == 0
assert CHOICE_SOURCE_MAP["kasama"].count("fake_generated") == 0
print("CHOICE_SOURCE_MAP built and verified.")
print(f"\nExample -- kaya: {list(zip(CHOICE_MAP['kaya'], CHOICE_SOURCE_MAP['kaya']))}")
print(f"Example -- ika:  {list(zip(CHOICE_MAP['ika'], CHOICE_SOURCE_MAP['ika']))}")

## Step 12: Assemble final test JSONL

`asymmetry_type` per homograph is computed from the FULL `TDR/` corpus
(`tdr-experiments/data/TDR/`, 10,065 files) -- same
grouped-count-ratio-`classify_asymmetry` logic used in
`analysis/05_iaa_and_tdr_stats.ipynb` / `analysis/07_split_distribution.ipynb`
(top-2 diacritized variant counts only, `Undiac_Count` excluded from the
ratio), run once here rather than per-row. This is a separate cell since
it's the one ~10k-file scan in this notebook and worth keeping isolated.

For each `df_test_instances` row: look up its 4 choices, shuffle them with
a seed derived from that row's own `id` (`random.Random(id).shuffle(...)`)
so the shuffle is reproducible per-instance without needing to track a
single global seed's consumption order across 8,041 rows, find `label` as
the shuffled index of `Gold_Variant`, and build the prompt string.

In [ ]:
def classify_asymmetry(ratio):
    if pd.isna(ratio):
        return None
    if 0.25 <= ratio <= 0.5:
        return "Workable Asymmetry"
    elif 0.1 <= ratio < 0.25:
        return "Moderate Asymmetry"
    elif ratio < 0.1:
        return "Extreme Asymmetry"
    else:
        return "Out of Range"


_tdr_candidates = [
    Path("tdr-experiments") / "data" / "TDR",
    Path("data") / "TDR",
]
TDR_DIR = next((p for p in _tdr_candidates if p.is_dir()), None)
assert TDR_DIR is not None, f"TDR not found in any of: {_tdr_candidates}"
print(f"TDR_DIR: {TDR_DIR.resolve()}")

_tdr_rows = []
_tdr_files = sorted(TDR_DIR.glob("*.txt"))
for path in _tdr_files:
    text = read_text_clean(path)
    for match in HOMOGRAPH_PATTERN.finditer(text):
        token = match.group(0)
        _tdr_rows.append({"Homograph": normalize_word(token), "Variant": token.lower()})

df_tdr_tokens_full = pd.DataFrame(_tdr_rows, columns=["Homograph", "Variant"])
print(f"Scanned {len(_tdr_files)} TDR file(s), {len(df_tdr_tokens_full)} homograph instance(s) found.")

_tdr_grouped = df_tdr_tokens_full.groupby(["Homograph", "Variant"]).size().reset_index(name="Count")

asymmetry_type_by_homograph = {}
for h in homographs:
    sub_all = _tdr_grouped.loc[_tdr_grouped["Homograph"] == h]
    diac_sub = (
        sub_all.loc[sub_all["Variant"] != h]
        .sort_values("Count", ascending=False)
        .reset_index(drop=True)
    )
    count_1 = int(diac_sub.loc[0, "Count"]) if len(diac_sub) > 0 else 0
    count_2 = int(diac_sub.loc[1, "Count"]) if len(diac_sub) > 1 else 0
    ratio = (count_2 / (count_1 + count_2)) if count_1 > 0 else float("nan")
    asymmetry_type_by_homograph[h] = classify_asymmetry(ratio)

print("\nAsymmetry_Type per homograph (from full TDR/):")
for h, atype in asymmetry_type_by_homograph.items():
    print(f"  {h:15s} {atype}")

In [ ]:
import random

eval_records = []
for _, row in df_test_instances.iterrows():
    homograph = row["Homograph"]
    choices = list(CHOICE_MAP[homograph])
    random.Random(row["Id"]).shuffle(choices)
    label = choices.index(row["Gold_Variant"])

    prompt = (
        f'How should the word {homograph} be written with the correct diacritic '
        f'marks in the sentence "{row["Context_Sentence"]}"?'
    )

    eval_records.append({
        "id": row["Id"],
        "homograph": homograph,
        "prompt": prompt,
        "choices": choices,
        "label": label,
        "gold_variant": row["Gold_Variant"],
        "asymmetry_type": asymmetry_type_by_homograph.get(homograph),
    })

print(f"Built {len(eval_records)} eval record(s).")

TEST_EVAL_JSONL = DATA_DIR / "test_eval.jsonl"

with open(TEST_EVAL_JSONL, "w", encoding="utf-8") as f:
    for record in eval_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(eval_records)} record(s) to {TEST_EVAL_JSONL.resolve()}")

## Step 13: Report

Re-reads `test_eval.jsonl` back from disk (not the in-memory
`eval_records`) to confirm the write/read round-trip, and spot-checks the
shuffle+label logic on a random sample.

In [ ]:
with open(TEST_EVAL_JSONL, encoding="utf-8") as f:
    reread_records = [json.loads(line) for line in f]

print(f"Total records exported: {len(reread_records)} (should be 8041)")

print("\n5 example records re-read from test_eval.jsonl:")
for i, record in enumerate(reread_records[:5]):
    print(f"\nRecord {i}:")
    print(record)

sample = random.Random(42).sample(reread_records, min(20, len(reread_records)))
labels_valid = all(0 <= r["label"] <= 3 for r in sample)
choices_match = all(r["choices"][r["label"]] == r["gold_variant"] for r in sample)
print(f"\nSpot-check on 20 random records:")
print(f"  All labels are valid indices (0-3): {labels_valid}")
print(f"  choices[label] == gold_variant for all 20: {choices_match}")

asymmetry_counts = Counter(r["asymmetry_type"] for r in reread_records)
print("\nPer-asymmetry-type instance counts:")
for atype, count in asymmetry_counts.most_common():
    print(f"  {atype}: {count}")
print(f"  Total: {sum(asymmetry_counts.values())} (should be {len(reread_records)})")

## Step 14: Build `benchmark.jsonl` by removing ambiguous same-homograph, different-variant sentences

If a sentence contains the SAME homograph twice with TWO DIFFERENT
`gold_variant` values (e.g. "kaya ... kayâ" once and "kaya ... káya"
elsewhere in the same sentence), a model scored on either occurrence sees
an identical `Context_Sentence` -- both instances are indistinguishable
from the model's input alone, so scoring either one is unreliable
regardless of which occurrence the record claims to be asking about.
Different homographs sharing a sentence are fine (they're different
words). The SAME homograph repeated with the SAME variant both times is
also fine (no ambiguity -- the right answer really is the same either
way). Only same-homograph + different-variant is the problem, and when it
happens the WHOLE sentence is dropped -- every record from it, not just
the conflicting ones -- since the ambiguous context is shared by all of
them.

Reuses `df_test_instances` (Filename/Sentence_Index) and `eval_records`
(built in the same row order via `iterrows()` in Step 12) directly from
memory rather than re-parsing `id` strings -- filenames themselves contain
underscores, which would make splitting `id` back into its components
ambiguous.

In [ ]:
from collections import defaultdict

assert len(df_test_instances) == len(eval_records)

records_with_context = []
for (_, row), record in zip(df_test_instances.iterrows(), eval_records):
    assert row["Id"] == record["id"]
    combined = dict(record)
    combined["Filename"] = row["Filename"]
    combined["Sentence_Index"] = row["Sentence_Index"]
    records_with_context.append(combined)

groups = defaultdict(list)
for rec in records_with_context:
    groups[(rec["Filename"], rec["Sentence_Index"])].append(rec)

ambiguous_examples = []  # (Filename, Sentence_Index, homograph, {variants})
for key, recs in groups.items():
    variants_by_homograph = defaultdict(set)
    for r in recs:
        variants_by_homograph[r["homograph"]].add(r["gold_variant"])
    for h, variants in variants_by_homograph.items():
        if len(variants) > 1:
            ambiguous_examples.append((key[0], key[1], h, variants))

ambiguous_keys = {(filename, sent_idx) for filename, sent_idx, _, _ in ambiguous_examples}

benchmark_records = [
    {k: v for k, v in rec.items() if k not in ("Filename", "Sentence_Index")}
    for rec in records_with_context
    if (rec["Filename"], rec["Sentence_Index"]) not in ambiguous_keys
]

print(f"test_eval.jsonl records:  {len(records_with_context)}")
print(f"Ambiguous sentences found: {len(ambiguous_keys)}")
print(f"benchmark_records:         {len(benchmark_records)}")

## Step 15: Export and report

`benchmark.jsonl` replaces `test_eval.jsonl` as the file used for all
subsequent model evaluation (zero-shot, few-shot, fine-tuned) across both
SEA-LION and Llama.

In [ ]:
BENCHMARK_JSONL = DATA_DIR / "benchmark.jsonl"

with open(BENCHMARK_JSONL, "w", encoding="utf-8") as f:
    for record in benchmark_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(benchmark_records)} record(s) to {BENCHMARK_JSONL.resolve()}")

print(f"\ntest_eval.jsonl: {len(records_with_context)} record(s)")
print(f"benchmark.jsonl: {len(benchmark_records)} record(s)")
print(f"Sentences excluded as ambiguous: {len(ambiguous_keys)}")

print("\nExample ambiguous sentences excluded (up to 5):")
for filename, sent_idx, h, variants in ambiguous_examples[:5]:
    print(f"  {filename} #{sent_idx}  homograph={h}  conflicting gold_variants={sorted(variants)}")

benchmark_per_homograph = Counter(r["homograph"] for r in benchmark_records)
print("\nPer-homograph count in benchmark.jsonl (most to fewest):")
for h, count in benchmark_per_homograph.most_common():
    print(f"  {h:15s} {count}")

below_threshold = {h: c for h, c in benchmark_per_homograph.items() if c < 5}
missing_after_filter = sorted(set(homographs) - set(benchmark_per_homograph))
print(f"\nHomograph(s) below 5 instances after filtering (flagged, not excluded):")
if below_threshold or missing_after_filter:
    for h, c in below_threshold.items():
        print(f"  {h:15s} {c}")
    for h in missing_after_filter:
        print(f"  {h:15s} 0 (no instances left after filtering)")
else:
    print("  None -- every homograph still has 5+ instances.")

benchmark_asymmetry_counts = Counter(r["asymmetry_type"] for r in benchmark_records)
print("\nPer-asymmetry-type instance counts in benchmark.jsonl:")
for atype, count in benchmark_asymmetry_counts.most_common():
    print(f"  {atype}: {count}")
print(f"  Total: {sum(benchmark_asymmetry_counts.values())} (should be {len(benchmark_records)})")